# Lakelink Fiber Provable Resilience: @data-analyst v4 candidate writers

**Owner:** @data-analyst. **Task:** P48-DA-V4-WRITERS-CHECKIN (CI gate `test_v4_candidate_writer_contains_literal_name`, BUG-QA-085).

**Objects written (only):**

| Object | Type | Origin task | Source |
| --- | --- | --- | --- |
| `cdm_tmforum.provable_resilience_intelligence.revenue_at_risk_v4_candidate` | Metric view (YAML 1.1) | BUG-QA-069 | frozen v3 `resilience_score_snapshot` + `service_inventory` + `customer` |

The DDL below was captured from the live object with `SHOW CREATE TABLE` on 2026-10-27, so this notebook reproduces exactly what is deployed.

**Freeze rules (DEC-PRI-V4-SHADOW-BUILD):**
* `write_enabled` defaults to `false`. With the default, the notebook only reads: it checks that the live definition matches the checked-in DDL and that the governed KPIs reproduce.
* Set `write_enabled=true` only in a @pm-approved write window. The write cell asserts the target ends in `_v4_candidate`.
* At v4 cutover (P30-DA-GENIE-MRC-PROTECTED-CUTOVER), repoint the `source` query to `resilience_score_snapshot_v4_candidate` here first, then re-run with `write_enabled=true`.

In [0]:
CATALOG_SCHEMA = "cdm_tmforum.provable_resilience_intelligence"
RAR_MV = f"{CATALOG_SCHEMA}.revenue_at_risk_v4_candidate"

dbutils.widgets.dropdown("write_enabled", "false", ["false", "true"], "Write enabled (@pm write window only)")
WRITE_ENABLED = dbutils.widgets.get("write_enabled") == "true"

# Freeze guard: this notebook may only write *_v4_candidate objects.
assert RAR_MV.endswith("_v4_candidate"), f"Refusing to write a non-candidate object: {RAR_MV}"

# Governed KPI baseline published by BUG-QA-069 (frozen v3 snapshot, data v22).
RAR_BASELINE = {"critical_service_count": 848, "at_risk_service_count": 2932, "total_revenue_at_risk": 48445093}
print(f"target={RAR_MV} write_enabled={WRITE_ENABLED}")

In [0]:
# Captured from the live object (SHOW CREATE TABLE, 2026-10-27). BUG-QA-069 governed bands:
# critical = overall_score < 40, at_risk = 40 <= overall_score < 60 (was score < 70 in the pipeline MV).
RAR_MV_DDL = r"""
CREATE OR REPLACE VIEW cdm_tmforum.provable_resilience_intelligence.revenue_at_risk_v4_candidate (
  customer_id COMMENT 'Lakelink Fiber customer identifier (FK to customer.customer_id)',
  customer_name COMMENT 'Customer legal/display name',
  customer_tier COMMENT 'Customer tier: platinum, gold, silver',
  industry COMMENT 'Customer industry vertical',
  region COMMENT 'Customer geographic region',
  customer_sla_tier COMMENT 'Customer SLA tier: premium_plus, premium, standard',
  service_type COMMENT 'Service type: wavelength, ethernet, dark_fiber, ip_vpn, managed_security',
  risk_level COMMENT 'Governed risk band: healthy (>=80), moderate (60-79), at_risk (40-59), critical (<40)',
  service_count COMMENT 'Distinct scored active services',
  customer_count COMMENT 'Distinct customers with scored active services',
  avg_resilience_score COMMENT 'Average composite resilience score (0-100, higher is better)',
  min_resilience_score COMMENT 'Lowest composite resilience score in the group',
  critical_service_count COMMENT 'Services with overall_score < 40 (governed critical KPI)',
  at_risk_service_count COMMENT 'Services with 40 <= overall_score < 60 (governed at_risk KPI; excludes critical). BUG-QA-069 fix: replaces pipeline score<70 definition',
  total_revenue_at_risk COMMENT 'Total monthly revenue at risk, USD (mrc_usd * (1 - overall_score/100))',
  critical_revenue_at_risk COMMENT 'Monthly revenue at risk from critical services (<40), USD',
  at_risk_revenue_at_risk COMMENT 'Monthly revenue at risk from at_risk services (40-59), USD',
  total_active_hazards COMMENT 'Sum of active hazards across services (service-hazard pairs, not distinct hazards)',
  total_shared_srlgs COMMENT 'Sum of shared SRLGs between working and protect paths across services')
COMMENT 'SHADOW v4 candidate (BUG-QA-069). Lakelink Fiber revenue at risk by customer, governed replacement for the pipeline gold_revenue_at_risk MV. Grain of source: one row per scored active service (resilience_score_snapshot) joined to service_inventory and customer. Fixes at_risk_services: governed KPI at_risk = 40 <= overall_score < 60 (was score < 70, which double-counted critical). critical = overall_score < 40. Sourced from the frozen v3 snapshot (data v22) so the fix is verifiable; repoint to resilience_score_snapshot_v4_candidate at v4 cutover.'
WITH METRICS
LANGUAGE YAML
AS
$$
version: 1.1

source: |-
  SELECT
    rss.service_id,
    rss.overall_score,
    rss.revenue_at_risk_usd,
    rss.active_hazards,
    rss.srlg_count,
    si.service_type,
    c.customer_id,
    c.customer_name,
    c.tier AS customer_tier,
    c.industry,
    c.region,
    c.sla_tier AS customer_sla_tier,
    CASE
      WHEN rss.overall_score >= 80 THEN 'healthy'
      WHEN rss.overall_score >= 60 THEN 'moderate'
      WHEN rss.overall_score >= 40 THEN 'at_risk'
      ELSE 'critical'
    END AS risk_level
  FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot rss
  JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory si
    ON rss.service_id = si.service_id
  JOIN cdm_tmforum.provable_resilience_intelligence.customer c
    ON si.customer_id = c.customer_id
  WHERE si.status = 'active'

comment: "SHADOW v4 candidate (BUG-QA-069). Lakelink Fiber revenue at risk by customer,\
  \ governed replacement for the pipeline gold_revenue_at_risk MV. Grain of source:\
  \ one row per scored active service (resilience_score_snapshot) joined to service_inventory\
  \ and customer. Fixes at_risk_services: governed KPI at_risk = 40 <= overall_score\
  \ < 60 (was score < 70, which double-counted critical). critical = overall_score\
  \ < 40. Sourced from the frozen v3 snapshot (data v22) so the fix is verifiable;\
  \ repoint to resilience_score_snapshot_v4_candidate at v4 cutover."

dimensions:
  - name: customer_id
    expr: customer_id
    comment: Lakelink Fiber customer identifier (FK to customer.customer_id)
    display_name: Customer ID

  - name: customer_name
    expr: customer_name
    comment: Customer legal/display name
    display_name: Customer Name

  - name: customer_tier
    expr: customer_tier
    comment: "Customer tier: platinum, gold, silver"
    display_name: Customer Tier

  - name: industry
    expr: industry
    comment: Customer industry vertical
    display_name: Industry

  - name: region
    expr: region
    comment: Customer geographic region
    display_name: Region

  - name: customer_sla_tier
    expr: customer_sla_tier
    comment: "Customer SLA tier: premium_plus, premium, standard"
    display_name: Customer SLA Tier

  - name: service_type
    expr: service_type
    comment: "Service type: wavelength, ethernet, dark_fiber, ip_vpn, managed_security"
    display_name: Service Type

  - name: risk_level
    expr: risk_level
    comment: "Governed risk band: healthy (>=80), moderate (60-79), at_risk (40-59),\
      \ critical (<40)"
    display_name: Risk Level

measures:
  - name: service_count
    expr: COUNT(DISTINCT service_id)
    comment: Distinct scored active services
    display_name: Services

  - name: customer_count
    expr: COUNT(DISTINCT customer_id)
    comment: Distinct customers with scored active services
    display_name: Customers

  - name: avg_resilience_score
    expr: "ROUND(AVG(overall_score), 1)"
    comment: "Average composite resilience score (0-100, higher is better)"
    display_name: Avg Resilience Score

  - name: min_resilience_score
    expr: "ROUND(MIN(overall_score), 1)"
    comment: Lowest composite resilience score in the group
    display_name: Min Resilience Score

  - name: critical_service_count
    expr: COUNT(DISTINCT service_id) FILTER (WHERE overall_score < 40)
    comment: Services with overall_score < 40 (governed critical KPI)
    display_name: Critical Services

  - name: at_risk_service_count
    expr: COUNT(DISTINCT service_id) FILTER (WHERE overall_score >= 40 AND overall_score
      < 60)
    comment: "Services with 40 <= overall_score < 60 (governed at_risk KPI; excludes\
      \ critical). BUG-QA-069 fix: replaces pipeline score<70 definition"
    display_name: At-Risk Services
    synonyms:
      - at risk services
      - at_risk_services

  - name: total_revenue_at_risk
    expr: SUM(revenue_at_risk_usd)
    comment: "Total monthly revenue at risk, USD (mrc_usd * (1 - overall_score/100))"
    display_name: Total Revenue at Risk
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact

  - name: critical_revenue_at_risk
    expr: SUM(revenue_at_risk_usd) FILTER (WHERE overall_score < 40)
    comment: "Monthly revenue at risk from critical services (<40), USD"
    display_name: Critical Revenue at Risk
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact

  - name: at_risk_revenue_at_risk
    expr: SUM(revenue_at_risk_usd) FILTER (WHERE overall_score >= 40 AND overall_score
      < 60)
    comment: "Monthly revenue at risk from at_risk services (40-59), USD"
    display_name: At-Risk Band Revenue at Risk
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact

  - name: total_active_hazards
    expr: SUM(active_hazards)
    comment: "Sum of active hazards across services (service-hazard pairs, not distinct\
      \ hazards)"
    display_name: Active Hazard Exposures

  - name: total_shared_srlgs
    expr: SUM(srlg_count)
    comment: Sum of shared SRLGs between working and protect paths across services
    display_name: Shared SRLGs
$$
""".strip()

In [0]:
import re

def _norm(sql: str) -> str:
    """Drop the CREATE prefix and collapse whitespace so only the body is compared."""
    body = re.sub(r"^\s*CREATE\s+(OR\s+REPLACE\s+)?VIEW\s+", "", sql.strip(), flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", body).strip()

live_ddl = spark.sql(f"SHOW CREATE TABLE {RAR_MV}").first()[0]
DDL_MATCHES_LIVE = _norm(live_ddl) == _norm(RAR_MV_DDL)
print(f"live definition matches checked-in DDL: {DDL_MATCHES_LIVE}")

In [0]:
kpis = spark.sql(f"""
  SELECT MEASURE(service_count) AS service_count,
         MEASURE(critical_service_count) AS critical_service_count,
         MEASURE(at_risk_service_count) AS at_risk_service_count,
         ROUND(MEASURE(total_revenue_at_risk)) AS total_revenue_at_risk
  FROM {RAR_MV}""").first().asDict()

for k, expected in RAR_BASELINE.items():
    print(f"{k}: live={kpis[k]} baseline={expected} match={int(kpis[k]) == expected}")
print(f"service_count: {kpis['service_count']}")

In [0]:
if WRITE_ENABLED:
    assert RAR_MV.endswith("_v4_candidate")
    spark.sql(RAR_MV_DDL)
    print(f"wrote {RAR_MV}")
else:
    print("write_enabled=false: no write. Set it to true only in a @pm-approved write window.")

## Genie PROVISIONAL disclosure (P68-DA-GENIE-PROVISIONAL-NOTE, contingency)

**Spec:** `apps/V4-KPI-COPY-SPEC.md` §7.5. **Decision:** `DEC-PRI-V4-073-SECOND-MISS-PARTIAL-CUTOVER` (decisions id 41), amended by id 42.

**Targets:** the two Lakelink Fiber demo Genie spaces. Both use resilience scores and revenue at risk, so both get the note.
* Service Assurance: `01f1cf046ba113e29bbc8a5aa9c35804`
* Sales: `01f1cf046d971e18a670736589cb94af`

**What gets added (idempotent, uses fixed ids):**
* A text instruction with the §7.5 general instruction verbatim, plus the banned list: no worse / declined / dropped / deteriorated / regression about scores, and no v4 deltas (56.8, 1,087, $54.4M/mo).
* The sample question `Which figures are provisional right now?`

**Gate:** the PATCH runs only when `write_enabled=true` **and** decisions id 41 has `status = 'decided'`. In every other case the cell only prints the diff. If the CEO approves the BUG-QA-073 writes (DEC-41 superseded), this is never applied.

**Rollback:** run `remove_provisional()` to drop the two fixed ids again (v4.1, or when DEC-41 is superseded).

In [0]:
import hashlib, json, re
from databricks.sdk import WorkspaceClient

_w = WorkspaceClient()
GENIE_SPACES = {"service_assurance": "01f1cf046ba113e29bbc8a5aa9c35804", "sales": "01f1cf046d971e18a670736589cb94af"}

# V4-KPI-COPY-SPEC §7.5, verbatim. Do not edit without @designer sign-off.
PROVISIONAL_INSTRUCTION = ('When an answer uses resilience scores, risk bands, critical or high risk counts, revenue at risk, or hazard exposure, end with: "Note: these figures are provisional. They use the v3 hazard model while the v4 hazard update completes. Topology, conduit and shared-risk figures are from the v4 build." Do not add the note to answers that use only topology, conduit, span or shared-risk data.')
BANNED_INSTRUCTION = ("Lakelink Fiber partial v4 cutover (DEC-41): never describe resilience scores as worse, declined, dropped, deteriorated or a regression. "
  "Never quote v4 score deltas or v4 hazard figures (61.5 to 56.8, 1,087 critical, $54.4M/mo); they are not live. "
  "Use only the live v3 figures from the governed metric views. Always refer to the company as Lakelink Fiber.")
SAMPLE_Q = "Which figures are provisional right now?"

def _id(label):
    return hashlib.md5(f"P68-DA-GENIE-PROVISIONAL-NOTE:{label}".encode()).hexdigest()
TI_ID, SQ_ID = _id("text_instruction"), _id("sample_question")  # d972eb8d..., 0e6eec55...

def _get(space_id):
    r = _w.api_client.do("GET", f"/api/2.0/genie/spaces/{space_id}", query={"include_serialized_space": "true"})
    return json.loads(r["serialized_space"])

def build_payload(current, add=True):
    s = json.loads(json.dumps(current))
    ti = s.setdefault("instructions", {}).setdefault("text_instructions", [])
    sq = s.setdefault("config", {}).setdefault("sample_questions", [])
    ti[:] = [t for t in ti if t.get("id") != TI_ID]
    sq[:] = [q for q in sq if q.get("id") != SQ_ID]
    if add:
        ti.append({"id": TI_ID, "content": [PROVISIONAL_INSTRUCTION + "\n", BANNED_INSTRUCTION]})
        sq.append({"id": SQ_ID, "question": [SAMPLE_Q]})
    ti.sort(key=lambda t: t["id"]); sq.sort(key=lambda q: q["id"])
    assert not re.search("lumen", json.dumps(s), re.I), "Branding guard: Lakelink Fiber only"
    assert s["data_sources"] == current["data_sources"], "data_sources must not change"
    return s

def _dec41_status():
    try:
        import psycopg
        ep = "projects/telco-provable-resilience-intelligence-project/branches/production/endpoints/primary"
        tok = _w.postgres.generate_database_credential(endpoint=ep).token
        with psycopg.connect(host="ep-summer-hill-d2s98kn6.database.us-east-1.cloud.databricks.com", dbname="databricks_postgres",
                             user=_w.current_user.me().user_name, password=tok, sslmode="require") as c:
            return c.execute("SELECT status FROM decisions WHERE id = 41").fetchone()[0]
    except Exception as e:
        print(f"Could not read DEC-41 status ({e}); treating as not decided.")
        return None

def _patch(space_id, payload):
    _w.api_client.do("PATCH", f"/api/2.0/genie/spaces/{space_id}", body={"serialized_space": json.dumps(payload)})

def remove_provisional():
    for name, sid in GENIE_SPACES.items():
        _patch(sid, build_payload(_get(sid), add=False)); print(f"removed PROVISIONAL note from {name}")

DEC41 = _dec41_status()
APPLY = WRITE_ENABLED and DEC41 == "decided"
print(f"DEC-41 status={DEC41} write_enabled={WRITE_ENABLED} -> apply={APPLY}")
for name, sid in GENIE_SPACES.items():
    cur_s = _get(sid); new_s = build_payload(cur_s)
    changed = new_s != cur_s
    print(f"{name}: text_instructions {len(cur_s.get('instructions',{}).get('text_instructions',[]))}->{len(new_s['instructions']['text_instructions'])}, "
          f"sample_questions {len(cur_s.get('config',{}).get('sample_questions',[]))}->{len(new_s['config']['sample_questions'])}, changed={changed}")
    if APPLY and changed:
        _patch(sid, new_s); print(f"  applied to {name}")
if not APPLY:
    print("Staged only: no Genie change. Apply needs write_enabled=true and DEC-41 decided.")

## schema_health_scorecard refresh (P80-DA-REFRESH-SCHEMA-HEALTH-SCORECARD)

Rebuilds `schema_health_scorecard` to cover **every PRI object** (MANAGED + VIEW + METRIC_VIEW), excluding temporary `*__v3_rollback` DEEP CLONE backups (DEC-46/49, dropped at v4.1). Before this refresh it covered 42 MANAGED tables, last assessed 2026-10-24.

**Rules**
* `sid_domain`: the latest non-superseded `demo_freeze_manifest` row, which is the governed @data-planner source. Otherwise v4/regen candidates inherit the domain of their v3 parent (`DOMAIN_OVERRIDES` below), and freeze/control tables map to Enterprise.
* `row_count` / `primary_origin`: live `COUNT(*)` and the dominant `origin` value (DERIVED if the table has no origin column). For views: NULL and `N/A_<type>`.
* `doc_completeness_pct`: share of commented columns in information_schema.columns.
* `fk_path_count` / `fk_pass_rate_pct`: from the `fk_integrity_report` metric view (child_table grain, PASS means integrity_pct >= 99).
* Grade: A = table comment + 100% docs + FK pass + (>=100 rows or a view); B = same but MANAGED with <100 rows; C = docs 90-99% or FK not passing; F = otherwise.

**Gate (P89-DA-SHS-FROZEN-WRITE-GUARD):** `schema_health_scorecard` is FROZEN in PRI-DEMO-FREEZE-v3 (restore_version 21), and DEC-48 forbids any commit to it. Under DEC-PRI-V4-SHADOW-BUILD this cell writes **only** `schema_health_scorecard_v4_candidate`, using the v3 table schema (a read). The write runs only when `write_enabled=true`, and it goes through `ml_freeze_guard.assert_writable` at runtime. Promoting the candidate to the live scorecard is a cutover-runbook / FREEZE-v4 manifest step owned by @devops/@data-planner, not this notebook. In every other case the cell prints the dry-run summary.

In [0]:
import re, datetime
import pandas as pd
import psycopg

import os, sys
# P89-DA-SHS-FROZEN-WRITE-GUARD: schema_health_scorecard is FROZEN in PRI-DEMO-FREEZE-v3 (DEC-48).
# Under DEC-PRI-V4-SHADOW-BUILD this cell writes ONLY the shadow candidate; the v3 table is read for its schema.
SHS_V3 = f"{CATALOG_SCHEMA}.schema_health_scorecard"
SHS_TARGET = f"{CATALOG_SCHEMA}.schema_health_scorecard_v4_candidate"
assert SHS_TARGET.endswith("_v4_candidate"), f"Refusing to write a non-candidate object: {SHS_TARGET}"
PRI_SCHEMA = CATALOG_SCHEMA.split(".")[1]
DOMAIN_OVERRIDES = {  # v4/regen candidates with no manifest row -> v3 parent domain
    "access_span_regen": "Resource", "access_span_geom_candidate": "Resource",
    "span_hazard_features_v4_candidate": "Resource", "path_diversity_conduit_v4_candidate": "Service",
    "kpi_delta_attribution_v4_candidate": "Enterprise",
}

tabs = spark.sql(f"""SELECT table_name, table_type, coalesce(comment,'') <> '' AS has_c
  FROM cdm_tmforum.information_schema.tables
  WHERE table_schema = '{PRI_SCHEMA}' AND table_name NOT LIKE '%\\_\\_v3\\_rollback'""").toPandas()
cols = spark.sql(f"""SELECT table_name, count(*) n, sum(CASE WHEN coalesce(comment,'') <> '' THEN 1 ELSE 0 END) d,
  max(CASE WHEN column_name = 'origin' THEN 1 ELSE 0 END) has_origin
  FROM cdm_tmforum.information_schema.columns WHERE table_schema = '{PRI_SCHEMA}' GROUP BY 1""").toPandas().set_index("table_name")
fkp = spark.sql(f"""SELECT child_table, count(*) paths, sum(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) pass
  FROM (SELECT child_table, fk_path, status, MEASURE(total_row_count) t FROM {CATALOG_SCHEMA}.fk_integrity_report GROUP BY ALL)
  GROUP BY 1""").toPandas().set_index("child_table")
man = spark.sql(f"""SELECT object_name, sid_domain FROM {CATALOG_SCHEMA}.demo_freeze_manifest
  WHERE sid_domain IS NOT NULL ORDER BY manifest_ts""").toPandas()
dom = dict(zip(man.object_name, man.sid_domain))  # latest manifest row wins

def _domain(n):
    if n in dom: return dom[n]
    if n in DOMAIN_OVERRIDES: return DOMAIN_OVERRIDES[n]
    parent = re.sub(r"(_v4_candidate|_regen_candidate|_regen|_geom_candidate|_candidate|_v4)$", "", n)
    if parent in dom: return dom[parent]  # v4/regen candidate inherits its v3 parent domain
    if re.search(r"freeze|manifest|gate|fingerprint|schema_health|registry", n): return "Enterprise"
    return None

assessed = datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
rows = []
for _, t in tabs.iterrows():
    n = t.table_name; c = cols.loc[n] if n in cols.index else None; rc = None
    if t.table_type == "MANAGED":
        if c is not None and c.has_origin:
            g = spark.table(f"{CATALOG_SCHEMA}.`{n}`").groupBy("origin").count().toPandas()
            rc = int(g["count"].sum()); po = g.sort_values("count", ascending=False).iloc[0]["origin"] if len(g) else None
        else:
            rc = spark.table(f"{CATALOG_SCHEMA}.`{n}`").count(); po = "DERIVED"
    else:
        po = "N/A_" + t.table_type
    doc = round(100.0 * c.d / c.n, 1) if c is not None and c.n else 0.0
    paths = int(fkp.loc[n, "paths"]) if n in fkp.index else 0
    pr = round(100.0 * fkp.loc[n, "pass"] / paths, 1) if paths else None
    if doc < 100 or not t.has_c: grade = "C" if doc >= 90 else "F"
    elif not (pr is None or pr == 100.0): grade = "C"
    elif t.table_type == "MANAGED" and rc is not None and rc < 100: grade = "B"
    else: grade = "A"
    rows.append(dict(table_name=n, sid_domain=_domain(n), row_count=rc, primary_origin=po, doc_completeness_pct=float(doc),
                     fk_path_count=paths, fk_pass_rate_pct=pr, readiness_grade=grade, table_type=t.table_type,
                     assessed_at=assessed, origin="DERIVED"))
new = pd.DataFrame(rows)
assert new.sid_domain.notna().all(), f"Unresolved sid_domain: {new[new.sid_domain.isna()].table_name.tolist()}"
assert not new.table_name.str.contains("lumen", case=False).any(), "Branding guard: Lakelink Fiber only"
print(f"objects={len(new)} types={new.table_type.value_counts().to_dict()}")
print(f"grades={new.readiness_grade.value_counts().to_dict()} domains={new.sid_domain.value_counts().to_dict()}")

def _cutover_done():
    try:
        ep = "projects/telco-provable-resilience-intelligence-project/branches/production/endpoints/primary"
        tok = _w.postgres.generate_database_credential(endpoint=ep).token
        with psycopg.connect(host="ep-summer-hill-d2s98kn6.database.us-east-1.cloud.databricks.com", dbname="databricks_postgres",
                             user=_w.current_user.me().user_name, password=tok, sslmode="require") as cx:
            return cx.execute("SELECT status FROM agent_tasks WHERE task_code = 'P81-DEVOPS-PARTIAL-CUTOVER-EXECUTE'").fetchone()[0] == "completed"
    except Exception as e:
        print(f"Could not read cutover status ({e}); treating as not done."); return False

print(f"cutover_execute_completed={_cutover_done()} (informational; promotion of the candidate to live is a cutover-runbook step, not done here)")

APPLY_SHS = WRITE_ENABLED  # shadow candidate only; never the FROZEN v3 table
if APPLY_SHS:
    _root = os.path.dirname(os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()))
    sys.path.append("/Workspace" + _root)
    from ml_freeze_guard import assert_writable
    assert_writable(spark, SHS_TARGET.split(".")[-1])  # runtime companion to tests/test_v4_frozen_write_guard.py
    target_schema = spark.table(SHS_V3).schema  # read-only use of the frozen v3 table
    spark.createDataFrame(new[[f.name for f in target_schema]], schema=target_schema).write.mode("overwrite").saveAsTable(SHS_TARGET)
    print(f"wrote {len(new)} rows to {SHS_TARGET}")
else:
    print(f"Staged only: no write. Set write_enabled=true in a @pm write window to build {SHS_TARGET}. {SHS_V3} is never written (DEC-48 freeze).")